In [ ]:
# BƯỚC 1: Mount Google Drive
# (Để đọc trực tiếp thư mục chứa text/ và image/ mà không cần zip)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# BƯỚC 2: Clone mã nguồn và cài đặt thư viện
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt

In [ ]:
# BƯỚC 3: Tạo cầu nối (Symlink) từ Google Drive sang thư mục code
# TODO: Bạn cần ĐỔI /content/drive/MyDrive/Tên_Thư_Mục_Data/data thành đúng đường dẫn của bạn
!rm -rf ./data
!ln -s /content/drive/MyDrive/Tên_Thư_Mục_Data/data ./data

# Kiểm tra xem symlink đã hoạt động chưa
!ls -la ./data

In [ ]:
# BƯỚC 3.5: Khởi tạo thư mục lưu trữ cho toàn bộ phiên chạy này
# Đảm bảo tất cả mô hình train trong hôm nay đều nằm chung một thư mục
import os
import datetime

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
drive_ckpt_path = f'/content/drive/MyDrive/SE365/checkpoints/{run_id}'
os.environ['DRIVE_CKPT'] = drive_ckpt_path

# Tạo thư mục con checkpoints và thư mục run_id bằng os.makedirs
os.makedirs(drive_ckpt_path, exist_ok=True)
print(f'Mọi checkpoint trong phiên này sẽ được lưu chung vào: {drive_ckpt_path}')

In [ ]:
# BƯỚC 4: Huấn luyện mô hình Text (XLM-RoBERTa)
!python main.py --mode train_text --epochs 1

# Lưu Checkpoint Text ngay lập tức
!cp ./checkpoints/* $DRIVE_CKPT/ && echo "Đã lưu checkpoint Text vào $DRIVE_CKPT"

In [ ]:
# BƯỚC 5: Huấn luyện mô hình Image (ConvNeXt)
!python main.py --mode train_image --epochs 1

# Lưu Checkpoint Image ngay lập tức
!cp ./checkpoints/* $DRIVE_CKPT/ && echo "Đã lưu checkpoint Image vào $DRIVE_CKPT"

In [ ]:
# BƯỚC 6: Huấn luyện mô hình Fusion (Kết hợp Text + Image)
!python main.py --mode train_fusion --epochs 1

# Lưu Checkpoint Fusion ngay lập tức
!cp ./checkpoints/* $DRIVE_CKPT/ && echo "Đã lưu checkpoint Fusion vào $DRIVE_CKPT"

In [ ]:
# BƯỚC 7: Đánh giá mô hình (Tính các metric MAE, MSE, RMSE)
!python test.py --mode train_fusion